# Ensemble + XGBoost — compare with/without extra pricer

Add an **XGBoost-on-embeddings** pricer and compare average error.

1. Train XGBoost: run `train_xgboost.py` from `week8` (once).
2. This notebook runs **Frontier-only** vs **Frontier + XGBoost** (0.8 / 0.2) on the test set.

In [ ]:
import os
import sys

# Resolve ensemble dir (this folder): walk up from cwd until we see xgboost_agent.py
_cwd = os.path.abspath(os.getcwd())
CONTRIB = _cwd
for _ in range(6):
    if os.path.isfile(os.path.join(_cwd, "xgboost_agent.py")):
        CONTRIB = _cwd
        break
    _parent = os.path.dirname(_cwd)
    if _parent == _cwd:
        break
    _cwd = _parent
WEEK8 = os.path.abspath(os.path.join(CONTRIB, "..", ".."))
if WEEK8 not in sys.path:
    sys.path.insert(0, WEEK8)
if CONTRIB not in sys.path:
    sys.path.insert(0, CONTRIB)

from dotenv import load_dotenv
load_dotenv(override=True)


In [ ]:
import chromadb
from agents.items import Item
from agents.evaluator import evaluate
from agents.frontier_agent import FrontierAgent
from xgboost_agent import XGBoostAgent

DB = os.path.join(WEEK8, "products_vectorstore")
DATASET = "ed-donner/items_lite"
EVAL_SIZE = 100

train, val, test = Item.from_hub(DATASET)
test = test[:EVAL_SIZE]
client = chromadb.PersistentClient(path=DB)
collection = client.get_or_create_collection("products")
frontier = FrontierAgent(collection)
xgb_agent = XGBoostAgent()

def predictor_frontier(item):
    return frontier.price(item.summary or item.title)

def predictor_frontier_plus_xgb(item):
    desc = item.summary or item.title
    return 0.8 * frontier.price(desc) + 0.2 * xgb_agent.price(desc)

print("--- Frontier only ---")
evaluate(predictor_frontier, test, size=EVAL_SIZE)
print("\n--- Frontier + XGBoost ---")
evaluate(predictor_frontier_plus_xgb, test, size=EVAL_SIZE)